# Set IE interactions

**EXPLORATORY** (CLAUDE.md §2: exempt from the TDD gate; the functions it calls,
`set_ie` and `reachable_pairs`, are tested in `tests/test_set_ie.py`).

Does a pair of circuit nodes have a joint effect different from the sum of its
parts? For each pair $A=\{a,b\}$ of mutually unreachable nodes and each prompt:

$$\Delta(A) = \mathrm{IE}(m;A) - \mathrm{IE}(m;a) - \mathrm{IE}(m;b)$$

all three from `set_ie` (IG along a path where only the set's nodes move; the
hive's per-node IEs move a whole site and are not comparable). To second order
$\Delta \approx \delta_a\delta_b\, v_a^\top H v_b$, so this measures how far
Marks' parallel-IG assumption is from holding on real feature pairs.

Reading: same-sign members with $|\mathrm{IE}(A)| < |\mathrm{IE}(a)+\mathrm{IE}(b)|$
are sub-additive (AND-like); $>$ is super-additive (OR / backup-like).

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))   # repo root, for `src`

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.attribution.nodes import all_set_ies, nde, read_node_hive, reachable_pairs, set_ie
from src.data.loaders import load_sva_examples
from src.metrics import get_metric
from src.models.suites import build_suite, load_model_and_saes, prepend_bos_for
from src.selection.aggregate import aggregate
from src.selection.threshold import select_by_threshold

NODE_DIR = Path("../outputs_sweep/node_attribution/77c6178e")   # rc_train, n=100, seed 42
NODE_THRESHOLD = 0.1      # Marks' circuit: the candidate nodes
N_EXAMPLES = 10           # prompts to compute on (first N of the hive's order)
IG_STEPS = 10
NDE_STEPS = 50            # blocking re-encodes a site: IG needs more steps (see tests)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = Path("../outputs/set_ie")
torch.manual_seed(42)

In [ ]:
if not (NODE_DIR / "manifest.json").is_file():
    raise FileNotFoundError(f"No node hive at {NODE_DIR}")

suite = build_suite("pythia-marks")
model, saes = load_model_and_saes(suite, DEVICE)
metric_fn = get_metric("sva_logit_diff")
bos = prepend_bos_for(saes)

# Same call and seed as node attribution, so example i here is example_idx i there.
examples = load_sva_examples("../data/marks/rc_train.json", 100, model,
                             use_min_length_only=True, seed=42)[:N_EXAMPLES]
toks = [(model.to_tokens(e.clean_prefix, prepend_bos=bos).to(DEVICE),
         model.to_tokens(e.patch_prefix, prepend_bos=bos).to(DEVICE)) for e in examples]

nodes = select_by_threshold(aggregate(read_node_hive(NODE_DIR), "none"), NODE_THRESHOLD)
cands = [tuple(int(v) for v in r) for r in
         nodes.loc[nodes["selected"], ["site_ordinal", "token_idx", "feature_idx"]]
              .itertuples(index=False)]
print(f"{len(cands)} candidate nodes, {len(examples)} prompts, seq len {toks[0][0].shape[1]}")

In [ ]:
# Reachability depends only on the model and the sequence length: compute once.
reach = reachable_pairs(model, suite, toks[0][0])
site = {s.ordinal: s for s in suite.sites}

def pair_type(a, b):
    sa, sb = site[a[0]], site[b[0]]
    if sa.ordinal == sb.ordinal:
        return "same site"
    if sa.block == sb.block and {sa.kind, sb.kind} == {"attn", "mlp"}:
        return "attn ∥ mlp"
    return "cross-layer"           # unreachable only through the causal mask

n_pairs = sum(1 for i, a in enumerate(cands) for b in cands[i + 1:]
              if ((a[:2], b[:2]) not in reach and (b[:2], a[:2]) not in reach))
print(f"{n_pairs} unreachable pairs x {len(examples)} prompts = {n_pairs * len(examples)} set IEs")

In [ ]:
rows = []
t0 = time.time()
for i, (ex, (clean, patch)) in enumerate(zip(examples, toks)):
    single = {a: set_ie(model, saes, suite, [a], clean, patch, metric_fn, ex, steps=IG_STEPS)
              for a in cands}
    joint = all_set_ies(model, saes, suite, cands, 2, clean, patch, metric_fn, ex,
                        steps=IG_STEPS, reach=reach)
    for (a, b), ie_ab in joint.items():
        rows.append({"example_idx": i, "a": a, "b": b, "pair_type": pair_type(a, b),
                     "ie_a": single[a], "ie_b": single[b], "ie_ab": ie_ab})
    print(f"prompt {i + 1}/{len(examples)}: {time.time() - t0:.0f}s elapsed")

per_prompt = pd.DataFrame(rows)
per_prompt["delta"] = per_prompt.ie_ab - per_prompt.ie_a - per_prompt.ie_b
OUT_DIR.mkdir(parents=True, exist_ok=True)
per_prompt.to_csv(OUT_DIR / "per_prompt.csv", index=False)

Mean-pool over prompts per pair. `rel` scales the interaction by the members'
own magnitudes; `consistency` is the fraction of prompts whose $\Delta$ has the
sign of the mean.

In [ ]:
g = per_prompt.groupby(["a", "b", "pair_type"])
pairs = g[["ie_a", "ie_b", "ie_ab", "delta"]].mean()
pairs["scale"] = g.apply(lambda d: (d.ie_a.abs() + d.ie_b.abs()).mean())
pairs["rel"] = pairs.delta / pairs.scale
pairs["consistency"] = g.apply(
    lambda d: (np.sign(d.delta) == np.sign(d.delta.mean())).mean())
same_sign = np.sign(pairs.ie_a) == np.sign(pairs.ie_b)
sub = pairs.ie_ab.abs() < (pairs.ie_a + pairs.ie_b).abs()
pairs["shape"] = np.where(~same_sign, "mixed sign", np.where(sub, "AND-like", "OR-like"))
pairs = pairs.reset_index()
pairs.to_csv(OUT_DIR / "pairs.csv", index=False)

abs_rel = pairs.rel.abs()
summary = pairs.assign(abs_rel=abs_rel).groupby("pair_type").agg(
    pairs=("abs_rel", "size"),
    median_abs_rel=("abs_rel", "median"),
    p90_abs_rel=("abs_rel", lambda s: s.quantile(0.9)),
    frac_over_10pct=("abs_rel", lambda s: (s > 0.1).mean()),
)
summary.round(3)

In [ ]:
COLORS = {"same site": "#2a78d6", "attn ∥ mlp": "#eb6834", "cross-layer": "#1baf7a"}
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3), facecolor=SURFACE)
for ax in (ax1, ax2):
    ax.set_facecolor(SURFACE)
    ax.grid(color=GRID, linewidth=0.6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(MUTED)
    ax.tick_params(colors=MUTED)

for kind, color in COLORS.items():
    d = pairs[pairs.pair_type == kind]
    if d.empty:
        continue
    x = np.sort(np.clip(d.rel.abs(), 1e-4, None))
    ax1.step(x, np.arange(1, len(x) + 1) / len(x), where="post", color=color,
             linewidth=2, label=f"{kind} ({len(d)})")
    ax2.scatter(d.ie_a + d.ie_b, d.ie_ab, s=36, color=color, alpha=0.75,
                edgecolor=SURFACE, linewidth=0.8, label=kind)
ax1.axvline(0.1, color=MUTED, linestyle="--", linewidth=1)
ax1.text(0.105, 0.03, "10%", color=MUTED, fontsize=9)
ax1.set_xscale("log")
ax1.set_xlabel("|Δ| / (|IE(a)| + |IE(b)|), mean over prompts", color=INK)
ax1.set_ylabel("fraction of pairs", color=INK)
ax1.set_title("How large is the interaction?", color=INK, loc="left")
ax1.legend(frameon=False, fontsize=9, labelcolor=INK)

lim = np.abs(np.r_[pairs.ie_a + pairs.ie_b, pairs.ie_ab]).max() * 1.05
ax2.plot([-lim, lim], [-lim, lim], color=MUTED, linestyle="--", linewidth=1)
ax2.text(lim * 0.55, lim * 0.42, "additive", color=MUTED, fontsize=9, rotation=45)
ax2.set_xlabel("IE(a) + IE(b)", color=INK)
ax2.set_ylabel("IE({a, b})", color=INK)
ax2.set_title("Joint vs summed effect, per pair", color=INK, loc="left")
ax2.legend(frameon=False, fontsize=9, labelcolor=INK)
fig.tight_layout()
fig.savefig("../figures/set-ie-interactions.png", dpi=150, facecolor=SURFACE)

Largest interactions, and whether they are consistent across prompts.

In [ ]:
cols = ["a", "b", "pair_type", "ie_a", "ie_b", "ie_ab", "delta", "rel", "consistency", "shape"]
pairs.reindex(pairs.delta.abs().sort_values(ascending=False).index)[cols].head(20).round(4)

## NDE: how much of a node's effect goes through another

For each ordered pair $a \rightsquigarrow b$ of circuit nodes,

$$\mathrm{NIE}(a \to b) = \mathrm{IE}(m;a) - \mathrm{NDE}(m;a,\{b\})$$

is the part of $a$'s effect that flows through $b$ (Pearl 2001); `share` is that
part over $\mathrm{IE}(m;a)$. Both terms use `NDE_STEPS`, so their discretisation
errors match. `via` says whether $b$ sits at $a$'s position or at a later one
(reached only through attention).

In [ ]:
med_pairs = [(a, b) for a in cands for b in cands if (a[:2], b[:2]) in reach]
print(f"{len(med_pairs)} ordered (a ⇝ b) pairs x {len(examples)} prompts")

med_rows = []
t0 = time.time()
for i, (ex, (clean, patch)) in enumerate(zip(examples, toks)):
    ie = {a: set_ie(model, saes, suite, [a], clean, patch, metric_fn, ex, steps=NDE_STEPS)
          for a in {a for a, _ in med_pairs}}
    for a, b in med_pairs:
        med_rows.append({
            "example_idx": i, "a": a, "b": b,
            "via": "same position" if a[1] == b[1] else "later position",
            "ie_a": ie[a],
            "nde": nde(model, saes, suite, [a], [b], clean, patch, metric_fn, ex,
                       steps=NDE_STEPS)})
    print(f"prompt {i + 1}/{len(examples)}: {time.time() - t0:.0f}s elapsed")

med = pd.DataFrame(med_rows)
med["mediated"] = med.ie_a - med.nde
med.to_csv(OUT_DIR / "nde_per_prompt.csv", index=False)

In [ ]:
g = med.groupby(["a", "b", "via"])
mp = g[["ie_a", "nde", "mediated"]].mean()
mp["share"] = mp.mediated / mp.ie_a
mp["consistency"] = g.apply(
    lambda d: (np.sign(d.mediated) == np.sign(d.mediated.mean())).mean())
mp = mp.reset_index()
mp.to_csv(OUT_DIR / "nde_pairs.csv", index=False)

mp.assign(abs_share=mp.share.abs()).groupby("via").agg(
    pairs=("abs_share", "size"),
    median_abs_share=("abs_share", "median"),
    p90_abs_share=("abs_share", lambda s: s.quantile(0.9)),
    frac_over_10pct=("abs_share", lambda s: (s > 0.1).mean()),
).round(3)

In [ ]:
VIA = {"same position": "#2a78d6", "later position": "#eb6834"}

def label(n):
    s = site[n[0]]
    return f"{s.kind}{'' if s.block < 0 else s.block}@{n[1]} " + ("err" if n[2] == -1 else f"#{n[2]}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6), facecolor=SURFACE)
for ax in (ax1, ax2):
    ax.set_facecolor(SURFACE)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(MUTED)
    ax.tick_params(colors=MUTED)
ax1.grid(color=GRID, linewidth=0.6)
ax2.grid(axis="x", color=GRID, linewidth=0.6)

for kind, color in VIA.items():
    d = mp[mp.via == kind]
    if d.empty:
        continue
    x = np.sort(np.clip(d.share.abs(), 1e-4, None))
    ax1.step(x, np.arange(1, len(x) + 1) / len(x), where="post", color=color,
             linewidth=2, label=f"{kind} ({len(d)})")
ax1.axvline(0.1, color=MUTED, linestyle="--", linewidth=1)
ax1.text(0.105, 0.03, "10%", color=MUTED, fontsize=9)
ax1.set_xscale("log")
ax1.set_xlabel("|IE(a) − NDE(a, {b})| / |IE(a)|, mean over prompts", color=INK)
ax1.set_ylabel("fraction of (a ⇝ b) pairs", color=INK)
ax1.set_title("How much of a's effect goes through b?", color=INK, loc="left")
ax1.legend(frameon=False, fontsize=9, labelcolor=INK)

# Mediators: total |mediated| effect each b carries, summed over its upstream a's.
hub = mp.assign(abs_med=mp.mediated.abs()).groupby("b").abs_med.sum().nlargest(15)[::-1]
ax2.barh([label(b) for b in hub.index], hub.values, height=0.6, color="#2a78d6",
         edgecolor=SURFACE, linewidth=2)
ax2.set_xlabel("Σ over a of |IE(a) − NDE(a, {b})|", color=INK)
ax2.set_title("Top 15 mediators", color=INK, loc="left")
ax2.tick_params(axis="y", labelsize=8, labelcolor=INK)
fig.tight_layout()
fig.savefig("../figures/nde-mediation.png", dpi=150, facecolor=SURFACE)

Largest single mediated effects.

In [ ]:
cols = ["a", "b", "via", "ie_a", "nde", "mediated", "share", "consistency"]
mp.reindex(mp.mediated.abs().sort_values(ascending=False).index)[cols].head(20).round(4)